# Case study: Preparing and summarizing diabetes risk survey data - Solution


# Full instructions and task list

In this case study, you will prepare and summarize a diabetes risk indicator dataset with pandas.

The notebook directly opens the local parquet file:

```python
data_path = Path("../../data/raw/diabetes.parquet")
df = pd.read_parquet(data_path)
```

Do not download the data in this exercise or load it from any alternative file. The public dataset links are included only for context and citation:

- UCI dataset page: https://archive.ics.uci.edu/dataset/891/cdc+diabetes+health+indicators
- Kaggle mirror: https://www.kaggle.com/datasets/alexteboul/diabetes-health-indicators-dataset

Each row represents one survey respondent. The variables include diabetes status, BMI, high blood pressure, high cholesterol, smoking, stroke, heart disease, physical activity, sex, age group, and general health.

## Parts and subtasks

### Part 1 - Load and inspect

- **1.1** Load the local parquet file into `df`.
- **1.2** Display the first rows and check the shape.
- **1.3** Inspect column names and data types.
- **1.4** Display descriptive statistics.
- **Bonus 1.A** Check missing values and unique values.
- **Bonus 1.B** Classify columns as binary, ordinal, continuous, or other.

### Part 2 - Select and filter

- **2.1** Create a smaller DataFrame called `df_health`.
- **2.2** Select respondents with BMI above 30.
- **2.3** Select respondents with both high blood pressure and high cholesterol.
- **2.4** Select respondents with diabetes.
- **Bonus 2.A** Build additional health-risk filters.
- **Bonus 2.B** Compare row counts across filtered dataframes.

### Part 3 - Clean names and labels

- **3.1** Rename columns to readable `snake_case` names.
- **3.2** Recode diabetes, sex, and general health values.
- **3.3** Check the recoded values.
- **Bonus 3.A** Convert selected columns to categorical dtype.

### Part 4 - Create derived variables

- **4.1** Create `bmi_category`.
- **4.2** Create `risk_score`.
- **4.3** Create `high_risk`.
- **4.4** Inspect the new columns.
- **Bonus 4.A** Create `risk_label`.
- **Bonus 4.B** Create `inactive_and_obese`.
- **Bonus 4.C** Create `cardiometabolic_risk`.

### Part 5 - Summarize the full dataset

- **5.1** Calculate overall summary statistics.
- **5.2** Summarize health indicators by diabetes status.
- **5.3** Summarize diabetes rate by BMI category.
- **5.4** Summarize BMI and risk score by sex and diabetes status.
- **Bonus 5.A** Add category shares and rounded values.
- **Bonus 5.B** Build a grouped summary by sex and BMI category.
- **Bonus 5.C** Keep only large groups and sort by diabetes rate.

### Part 6 - Optional transform challenge

- **6.1** Add the average risk score within each BMI category to every row.
- **6.2** Create `risk_above_bmi_category_average`.
- **6.3** Add the median BMI within each sex group to every row.
- **6.4** Compare `agg()` and `transform()` in one or two sentences.


# Preparations


In [1]:
from pathlib import Path

import pandas as pd
from pandas.api.types import CategoricalDtype

DATA_PATH = Path("../../data/raw/diabetes.parquet")

pd.set_option("display.max_columns", 500)

# Part 1 - Load and inspect


## 1.1 Load the local parquet file into `df`.

Load `../../data/raw/diabetes.parquet` with `pd.read_parquet()` and store the result in a DataFrame called `df`.


In [2]:
df = pd.read_parquet(DATA_PATH)
df.head()

,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,Veggies,HvyAlcoholConsump,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income,Diabetes_binary
0,1.0,1.0,1.0,40.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,5.0,18.0,15.0,1.0,0.0,9.0,4.0,3.0,0.0
1,0.0,0.0,0.0,25.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,3.0,0.0,0.0,0.0,0.0,7.0,6.0,1.0,0.0
2,1.0,1.0,1.0,28.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,5.0,30.0,30.0,1.0,0.0,9.0,4.0,8.0,0.0
3,1.0,0.0,1.0,27.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,2.0,0.0,0.0,0.0,0.0,11.0,3.0,6.0,0.0
4,1.0,1.0,1.0,24.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,2.0,3.0,0.0,0.0,0.0,11.0,5.0,4.0,0.0


## 1.2 Display the first rows and check the shape.

Display the first five rows and check how many rows and columns the dataset has.


In [3]:
print(df.shape)
df.head()

(253680, 22)


,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,Veggies,HvyAlcoholConsump,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income,Diabetes_binary
0,1.0,1.0,1.0,40.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,5.0,18.0,15.0,1.0,0.0,9.0,4.0,3.0,0.0
1,0.0,0.0,0.0,25.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,3.0,0.0,0.0,0.0,0.0,7.0,6.0,1.0,0.0
2,1.0,1.0,1.0,28.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,5.0,30.0,30.0,1.0,0.0,9.0,4.0,8.0,0.0
3,1.0,0.0,1.0,27.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,2.0,0.0,0.0,0.0,0.0,11.0,3.0,6.0,0.0
4,1.0,1.0,1.0,24.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,2.0,3.0,0.0,0.0,0.0,11.0,5.0,4.0,0.0


## 1.3 Inspect column names and data types.

Print the column names and use `.info()` to inspect the data types.


In [4]:
print(df.columns.tolist())
df.info()

['HighBP', 'HighChol', 'CholCheck', 'BMI', 'Smoker', 'Stroke', 'HeartDiseaseorAttack', 'PhysActivity', 'Fruits', 'Veggies', 'HvyAlcoholConsump', 'AnyHealthcare', 'NoDocbcCost', 'GenHlth', 'MentHlth', 'PhysHlth', 'DiffWalk', 'Sex', 'Age', 'Education', 'Income', 'Diabetes_binary']
<class 'pandas.DataFrame'>
Index: 253680 entries, 0 to 253679
Data columns (total 22 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   HighBP                253680 non-null  float64
 1   HighChol              253680 non-null  float64
 2   CholCheck             253680 non-null  float64
 3   BMI                   253680 non-null  float64
 4   Smoker                253680 non-null  float64
 5   Stroke                253680 non-null  float64
 6   HeartDiseaseorAttack  253680 non-null  float64
 7   PhysActivity          253680 non-null  float64
 8   Fruits                253680 non-null  float64
 9   Veggies               253680 non-null  float64
 

## 1.4 Display descriptive statistics.

Use `.describe()` to get a first numerical summary of the dataset.


In [5]:
df.describe()

,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,Veggies,HvyAlcoholConsump,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income,Diabetes_binary
count,253680.000000,253680.000000,253680.000000,253680.000000,253680.000000,253680.000000,253680.000000,253680.000000,253680.000000,253680.000000,253680.000000,253680.000000,253680.000000,253680.000000,253680.000000,253680.000000,253680.000000,253680.000000,253680.000000,253680.000000,253680.000000,253680.000000
mean,0.429001,0.424121,0.962670,28.382364,0.443169,0.040571,0.094186,0.756544,0.634256,0.811420,0.056197,0.951053,0.084177,2.511392,3.184772,4.242081,0.168224,0.440342,8.032119,5.050434,6.053875,0.139333
std,0.494934,0.494210,0.189571,6.608694,0.496761,0.197294,0.292087,0.429169,0.481639,0.391175,0.230302,0.215759,0.277654,1.068477,7.412847,8.717951,0.374066,0.496429,3.054220,0.985774,2.071148,0.346294
min,0.000000,0.000000,0.000000,12.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000,0.000000
25%,0.000000,0.000000,1.000000,24.000000,0.000000,0.000000,0.000000,1.000000,0.000000,1.000000,0.000000,1.000000,0.000000,2.000000,0.000000,0.000000,0.000000,0.000000,6.000000,4.000000,5.000000,0.000000
50%,0.000000,0.000000,1.000000,27.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000,0.000000,1.000000,0.000000,2.000000,0.000000,0.000000,0.000000,0.000000,8.000000,5.000000,7.000000,0.000000
75%,1.000000,1.000000,1.000000,31.000000,1.000000,0.000000,0.000000,1.000000,1.000000,1.000000,0.000000,1.000000,0.000000,3.000000,2.000000,3.000000,0.000000,1.000000,10.000000,6.000000,8.000000,0.000000
max,1.000000,1.000000,1.000000,98.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,5.000000,30.000000,30.000000,1.000000,1.000000,13.000000,6.000000,8.000000,1.000000


## Bonus 1.A Check missing values and unique values.

Create a small table with the number of missing values and the number of unique values per column.


In [6]:
pd.DataFrame(
    {
        "missing_values": df.isna().sum(),
        "unique_values": df.nunique(),
    }
)

,missing_values,unique_values
HighBP,0,2
HighChol,0,2
CholCheck,0,2
BMI,0,84
Smoker,0,2
Stroke,0,2
HeartDiseaseorAttack,0,2
PhysActivity,0,2
Fruits,0,2
Veggies,0,2


## Bonus 1.B Classify columns by variable type.

Create a rough classification of columns as binary, ordinal, continuous, or other.


In [7]:
(
    pd.Series("other", index=df.columns, name="suggested_type")
    .mask(df.nunique() == 2, "binary")
    .mask(df.columns.isin(["Age", "Education", "Income", "GenHlth"]), "ordinal")
    .mask(df.columns.isin(["BMI", "MentHlth", "PhysHlth"]), "continuous")
    .to_frame()
)

,suggested_type
HighBP,binary
HighChol,binary
CholCheck,binary
BMI,continuous
Smoker,binary
Stroke,binary
HeartDiseaseorAttack,binary
PhysActivity,binary
Fruits,binary
Veggies,binary


## Huddle after Part 1

> Checkpoint: `df` should exist, contain the parquet data, and have the expected health indicator columns.


# Part 2 - Select and filter


## 2.1 Create a smaller DataFrame called `df_health`.

Keep only the columns needed for the case study:

```text
Diabetes_binary, BMI, HighBP, HighChol, Smoker, Stroke, HeartDiseaseorAttack,
PhysActivity, Sex, Age, GenHlth
```


In [8]:
health_columns = [
    "Diabetes_binary",
    "BMI",
    "HighBP",
    "HighChol",
    "Smoker",
    "Stroke",
    "HeartDiseaseorAttack",
    "PhysActivity",
    "Sex",
    "Age",
    "GenHlth",
]

df_health = df.loc[:, health_columns].copy()
df_health.head()

,Diabetes_binary,BMI,HighBP,HighChol,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Sex,Age,GenHlth
0,0.0,40.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,9.0,5.0
1,0.0,25.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,7.0,3.0
2,0.0,28.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,9.0,5.0
3,0.0,27.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,11.0,2.0
4,0.0,24.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,11.0,2.0


## 2.2 Select respondents with BMI above 30.

Create a DataFrame called `df_bmi_above_30`.


In [9]:
df_bmi_above_30 = df_health.loc[df_health["BMI"] > 30]
print(df_bmi_above_30.shape)
df_bmi_above_30.head()

(73278, 11)


,Diabetes_binary,BMI,HighBP,HighChol,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Sex,Age,GenHlth
0,0.0,40.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,9.0,5.0
11,0.0,34.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,10.0,3.0
14,0.0,33.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,4.0,4.0
15,0.0,33.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,6.0,2.0
21,0.0,38.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,13.0,5.0


## 2.3 Select respondents with both high blood pressure and high cholesterol.

Create a DataFrame called `df_bp_and_chol`.


In [10]:
df_bp_and_chol = df_health.loc[(df_health["HighBP"] == 1) & (df_health["HighChol"] == 1)]
print(df_bp_and_chol.shape)
df_bp_and_chol.head()

(64660, 11)


,Diabetes_binary,BMI,HighBP,HighChol,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Sex,Age,GenHlth
0,0.0,40.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,9.0,5.0
2,0.0,28.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,9.0,5.0
4,0.0,24.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,11.0,2.0
5,0.0,25.0,1.0,1.0,1.0,0.0,0.0,1.0,1.0,10.0,2.0
7,0.0,25.0,1.0,1.0,1.0,0.0,0.0,1.0,0.0,11.0,3.0


## 2.4 Select respondents with diabetes.

Create a DataFrame called `df_with_diabetes`.


In [11]:
df_with_diabetes = df_health.loc[df_health["Diabetes_binary"] == 1]
print(df_with_diabetes.shape)
df_with_diabetes.head()

(35346, 11)


,Diabetes_binary,BMI,HighBP,HighChol,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Sex,Age,GenHlth
8,1.0,30.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0,9.0,5.0
10,1.0,25.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,13.0,3.0
13,1.0,28.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,11.0,4.0
17,1.0,23.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,7.0,2.0
23,1.0,27.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,13.0,1.0


## Bonus 2.A Build additional health-risk filters.

Create three additional filtered DataFrames:

1. Respondents with BMI above 30 and no physical activity.
2. Respondents with either stroke or heart disease.
3. A compact high-risk DataFrame with only diabetes, BMI, high blood pressure, high cholesterol, stroke, and heart disease columns.


In [12]:
df_obese_inactive = df_health.loc[
    (df_health["BMI"] > 30) & (df_health["PhysActivity"] == 0)
]

df_cardio_history = df_health.loc[
    (df_health["Stroke"] == 1) | (df_health["HeartDiseaseorAttack"] == 1)
]
high_risk_columns = [
    "Diabetes_binary",
    "BMI",
    "HighBP",
    "HighChol",
    "Stroke",
    "HeartDiseaseorAttack",
]
df_high_risk_raw = df_health.loc[
    (df_health["HighBP"] == 1) & (df_health["HighChol"] == 1) & (df_health["BMI"] > 30),
    high_risk_columns,
]

## Bonus 2.B Compare row counts across filtered dataframes

e.g.: df_health, df_bmi_above_30, df_bp_and_chol, df_with_diabetes, df_obese_inactive, df_cardio_history, df_high_risk_raw,

In [13]:
filter_counts = {
        "all_respondents": len(df_health),
        "df_bmi_above_30": len(df_bmi_above_30),
        "df_bp_and_chol": len(df_bp_and_chol),
        "df_with_diabetes": len(df_with_diabetes),
        "df_obese_inactive": len(df_obese_inactive),
        "df_cardio_history": len(df_cardio_history),
        "df_high_risk_raw": len(df_high_risk_raw),
    }
filter_counts

{'all_respondents': 253680,
 'df_bmi_above_30': 73278,
 'df_bp_and_chol': 64660,
 'df_with_diabetes': 35346,
 'df_obese_inactive': 24570,
 'df_cardio_history': 30248,
 'df_high_risk_raw': 25599}

## Huddle after Part 2

> Checkpoint: `df_health` should contain only the selected health columns, and the filtered DataFrames should have fewer rows than the full dataset.


# Part 3 - Clean names and labels


## 3.1 Rename columns

Rename the columns in `df_health` to shorter and more readable names: 

"Diabetes_binary": "diabetes"
"BMI": "bmi"
"HighBP": "high_bp"
"HighChol": "high_chol"
"Smoker": "smoker"
"Stroke": "stroke"
"HeartDiseaseorAttack": "heart_disease"
"PhysActivity": "physical_activity"
"Sex": "sex"
"Age": "age"
"GenHlth": "general_health"

In [14]:
rename_map = {
    "Diabetes_binary": "diabetes",
    "BMI": "bmi",
    "HighBP": "high_bp",
    "HighChol": "high_chol",
    "Smoker": "smoker",
    "Stroke": "stroke",
    "HeartDiseaseorAttack": "heart_disease",
    "PhysActivity": "physical_activity",
    "Sex": "sex",
    "Age": "age",
    "GenHlth": "general_health",
}
df_health = df_health.rename(columns=rename_map)
df_health.head()

,diabetes,bmi,high_bp,high_chol,smoker,stroke,heart_disease,physical_activity,sex,age,general_health
0,0.0,40.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,9.0,5.0
1,0.0,25.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,7.0,3.0
2,0.0,28.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,9.0,5.0
3,0.0,27.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,11.0,2.0
4,0.0,24.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,11.0,2.0


## 3.2 Recode diabetes, sex, and general health values.

Replace selected encoded values with readable labels:

```text
diabetes: 0 -> no_diabetes, 1 -> diabetes
sex: 0 -> female, 1 -> male
general_health: 1 -> excellent, 2 -> very_good, 3 -> good, 4 -> fair, 5 -> poor
```


In [15]:
df_health["diabetes"] = df_health["diabetes"].replace({0: "no_diabetes", 1: "diabetes"})
df_health["sex"] = df_health["sex"].replace({0: "female", 1: "male"})
df_health["general_health"] = df_health["general_health"].replace(
    {
        1: "excellent",
        2: "very_good",
        3: "good",
        4: "fair",
        5: "poor",
    }
)

df_health.head()

,diabetes,bmi,high_bp,high_chol,smoker,stroke,heart_disease,physical_activity,sex,age,general_health
0,no_diabetes,40.0,1.0,1.0,1.0,0.0,0.0,0.0,female,9.0,poor
1,no_diabetes,25.0,0.0,0.0,1.0,0.0,0.0,1.0,female,7.0,good
2,no_diabetes,28.0,1.0,1.0,0.0,0.0,0.0,0.0,female,9.0,poor
3,no_diabetes,27.0,1.0,0.0,0.0,0.0,0.0,1.0,female,11.0,very_good
4,no_diabetes,24.0,1.0,1.0,0.0,0.0,0.0,1.0,female,11.0,very_good


## 3.3 Check the recoded values.

Use `value_counts()` to check the unique values and frequencies for the recoded columns.


In [16]:
for column in ["diabetes", "sex", "general_health"]:
    print() # just a new line
    print(column)
    print(df_health[column].value_counts(dropna=False))


diabetes
diabetes
no_diabetes    218334
diabetes        35346
Name: count, dtype: int64

sex
sex
female    141974
male      111706
Name: count, dtype: int64

general_health
general_health
very_good    89084
good         75646
excellent    45299
fair         31570
poor         12081
Name: count, dtype: int64


## Bonus 3.A Convert selected columns to categorical dtype.

Convert `diabetes`, `sex`, and `general_health` to categorical dtype. Make `general_health` ordered from `excellent` to `poor`.


In [17]:
general_health_type = CategoricalDtype(
    categories=["excellent", "very_good", "good", "fair", "poor"],
    ordered=True,
)

df_health["diabetes"] = df_health["diabetes"].astype("category")
df_health["sex"] = df_health["sex"].astype("category")
df_health["general_health"] = df_health["general_health"].astype(general_health_type)

df_health[["diabetes", "sex", "general_health"]].dtypes

diabetes          category
sex               category
general_health    category
dtype: object

## Huddle after Part 3

> Checkpoint: `df_health` should now have readable column names and readable labels for diabetes, sex, and general health.


# Part 4 - Create derived variables


## 4.1 Create `bmi_category`.

Create a new column with four BMI categories:

```text
BMI < 18.5          -> underweight
18.5 <= BMI < 25    -> normal
25 <= BMI < 30      -> overweight
BMI >= 30           -> obese
```


In [18]:
df_health["bmi_category"] = "unknown"
df_health.loc[df_health["bmi"] < 18.5, "bmi_category"] = "underweight"
df_health.loc[
    (df_health["bmi"] >= 18.5) & (df_health["bmi"] < 25),
    "bmi_category",
] = "normal"
df_health.loc[
    (df_health["bmi"] >= 25) & (df_health["bmi"] < 30),
    "bmi_category",
] = "overweight"
df_health.loc[df_health["bmi"] >= 30, "bmi_category"] = "obese"

df_health["bmi_category"] = df_health["bmi_category"].astype("category")

df_health["bmi_category"].value_counts(sort=False)

bmi_category
normal         68953
obese          87851
overweight     93749
underweight     3127
Name: count, dtype: int64

## 4.2 Create `risk_score`.

Create a simple risk score by summing these binary indicators:

```text
high_bp, high_chol, smoker, stroke, heart_disease
```


In [19]:
risk_columns = ["high_bp", "high_chol", "smoker", "stroke", "heart_disease"]
df_health["risk_score"] = df_health[risk_columns].sum(axis=1)

df_health[[*risk_columns, "risk_score"]].head()

,high_bp,high_chol,smoker,stroke,heart_disease,risk_score
0,1.0,1.0,1.0,0.0,0.0,3.0
1,0.0,0.0,1.0,0.0,0.0,1.0
2,1.0,1.0,0.0,0.0,0.0,2.0
3,1.0,0.0,0.0,0.0,0.0,1.0
4,1.0,1.0,0.0,0.0,0.0,2.0


## 4.3 Create `high_risk`.

Create a boolean column called `high_risk` that is `True` when `risk_score >= 3`.


In [20]:
df_health["high_risk"] = df_health["risk_score"] >= 3
df_health["high_risk"].value_counts()

high_risk
False    207181
True      46499
Name: count, dtype: int64

## 4.4 Inspect the new columns.

Display the first rows of `bmi`, `bmi_category`, `risk_score`, and `high_risk`.


In [21]:
df_health[["bmi", "bmi_category", "risk_score", "high_risk"]].head()

,bmi,bmi_category,risk_score,high_risk
0,40.0,obese,3.0,True
1,25.0,overweight,1.0,False
2,28.0,overweight,2.0,False
3,27.0,overweight,1.0,False
4,24.0,normal,2.0,False


## Bonus 4.A Create `risk_label`.

Create a label from the risk score:

```text
0-1 risk factors -> low
2 risk factors   -> medium
3+ risk factors  -> high
```


In [22]:
df_health["risk_label"] = "low"
df_health.loc[df_health["risk_score"] == 2, "risk_label"] = "medium"
df_health.loc[df_health["risk_score"] >= 3, "risk_label"] = "high"

df_health["risk_label"].value_counts()

risk_label
low       142899
medium     64282
high       46499
Name: count, dtype: int64

## Bonus 4.B Create `inactive_and_obese`.

Create a boolean column that is `True` for respondents who are obese and report no physical activity.


In [23]:
df_health["inactive_and_obese"] = (df_health["bmi_category"] == "obese") & (
    df_health["physical_activity"] == 0
)

df_health["inactive_and_obese"].value_counts()

inactive_and_obese
False    225361
True      28319
Name: count, dtype: int64

## Bonus 4.C Create `cardiometabolic_risk`.

Create a boolean column that is `True` if the respondent has at least two of these indicators:

```text
high_bp, high_chol, obese BMI category, heart_disease
```


In [24]:
df_health["cardiometabolic_risk_score"] = (
    df_health["high_bp"]
    + df_health["high_chol"]
    + (df_health["bmi_category"] == "obese").astype(int)
    + df_health["heart_disease"]
)
df_health["cardiometabolic_risk"] = df_health["cardiometabolic_risk_score"] >= 2

df_health[
    ["cardiometabolic_risk_score", "cardiometabolic_risk"]
].value_counts().sort_index()

cardiometabolic_risk_score  cardiometabolic_risk
0.0                         False                   74688
1.0                         False                   76425
2.0                         True                    62377
3.0                         True                    33775
4.0                         True                     6415
Name: count, dtype: int64

## Huddle after Part 4

> Checkpoint: `df_health` should now include `bmi_category`, `risk_score`, and `high_risk`. Bonus columns may also be present.


# Part 5 - Summarize the dataset


## 5.1 Calculate overall summary statistics.

Calculate the average BMI, median BMI, average risk score, and median risk score.


In [25]:
df_health.agg({
    "bmi": ["mean", "median"],
    "risk_score": ["mean", "median"]
})

,bmi,risk_score
mean,28.382364,1.431047
median,27.000000,1.000000


## 5.2 Summarize health indicators by diabetes status.

Group by `diabetes` and summarize BMI, high blood pressure, high cholesterol, and risk score.


In [26]:
df_health.groupby("diabetes", observed=True).agg(
    respondents=("diabetes", "size"),
    average_bmi=("bmi", "mean"),
    median_bmi=("bmi", "median"),
    high_bp_rate=("high_bp", "mean"),
    high_chol_rate=("high_chol", "mean"),
    average_risk_score=("risk_score", "mean"),
)

,respondents,average_bmi,median_bmi,high_bp_rate,high_chol_rate,average_risk_score
diabetes,,,,,,
diabetes,35346,31.944011,31.0,0.752674,0.670118,2.256351
no_diabetes,218334,27.805770,27.0,0.376602,0.384297,1.297439


## 5.3 Summarize diabetes rate by BMI category.

Group by `bmi_category` and diabetes rate


In [27]:
df_health["has_diabetes"] = df_health["diabetes"] == "diabetes"

df_health.groupby("bmi_category", observed=True).agg(
    {"has_diabetes": "mean"}
)

,has_diabetes
bmi_category,
normal,0.056966
obese,0.233998
overweight,0.114049
underweight,0.054045


## 5.4 Summarize BMI and risk score by sex and diabetes status.

Group by `sex` and `diabetes`, then calculate average BMI and average risk score.


In [28]:
df_health.groupby(["sex", "diabetes"], observed=True).agg(
        respondents=("diabetes", "size"),
        average_bmi=("bmi", "mean"),
        average_risk_score=("risk_score", "mean"),
    ).round(2)

respondents  average_bmi  average_risk_score
sex    diabetes                                                 
female diabetes           18411        32.51                2.16
       no_diabetes       123563        27.48                1.21
male   diabetes           16935        31.33                2.36
       no_diabetes        94771        28.23                1.42

## Bonus 5.A Add category shares and rounded values.

Calculate the share of respondents in each BMI category.


In [29]:
df_health["bmi_category"].value_counts(normalize=True).sort_index().round(2)

bmi_category
normal         0.27
obese          0.35
overweight     0.37
underweight    0.01
Name: proportion, dtype: float64

## Bonus 5.B Build a grouped summary by sex and BMI category.

Group by `sex` and `bmi_category`, then calculate respondent count, diabetes rate, average BMI, and average risk score.


In [30]:
(
    df_health.groupby(["sex", "bmi_category"], observed=True)
    .agg(
        respondents=("diabetes", "size"),
        diabetes_rate=("has_diabetes", "mean"),
        average_bmi=("bmi", "mean"),
        average_risk_score=("risk_score", "mean"),
    )
    .round(2)
)

respondents  diabetes_rate  average_bmi  \
sex    bmi_category                                            
female normal              44947           0.05        22.06   
       obese               47347           0.23        35.72   
       overweight          47219           0.11        26.86   
       underweight          2461           0.05        17.36   
male   normal              24006           0.08        22.74   
       obese               40504           0.23        34.43   
       overweight          46530           0.12        26.96   
       underweight           666           0.08        17.10   

                     average_risk_score  
sex    bmi_category                      
female normal                      1.04  
       obese                       1.58  
       overweight                  1.36  
       underweight                 1.22  
male   normal                      1.29  
       obese                       1.77  
       overweight                  1.51  
       underweight                 1.45

## Bonus 5.C Keep only large groups and sort by diabetes rate.

Keep only groups with at least 1,000 respondents and sort by diabetes rate in descending order.


In [31]:
(
    df_health.groupby(["sex", "bmi_category"], observed=True)
    .agg(
        respondents=("diabetes", "size"),
        diabetes_rate=("has_diabetes", "mean"),
        average_bmi=("bmi", "mean"),
        average_risk_score=("risk_score", "mean"),
    )
    .round(2)
    .loc[lambda df: df["respondents"] >= 1_000]
    .sort_values("diabetes_rate", ascending=False)
)

respondents  diabetes_rate  average_bmi  \
sex    bmi_category                                            
female obese               47347           0.23        35.72   
male   obese               40504           0.23        34.43   
       overweight          46530           0.12        26.96   
female overweight          47219           0.11        26.86   
male   normal              24006           0.08        22.74   
female normal              44947           0.05        22.06   
       underweight          2461           0.05        17.36   

                     average_risk_score  
sex    bmi_category                      
female obese                       1.58  
male   obese                       1.77  
       overweight                  1.51  
female overweight                  1.36  
male   normal                      1.29  
female normal                      1.04  
       underweight                 1.22

## Huddle after Part 5

> Checkpoint: You should have groupwise summaries that reveal how diabetes rate and risk indicators differ across groups.


# Part 6 - Optional transform challenge


## 6.1 Add the average risk score within each BMI category to every row.

Use `groupby().transform()` to create `avg_risk_score_in_bmi_category`.


In [32]:
df_health["avg_risk_score_in_bmi_category"] = df_health.groupby(
    "bmi_category",
    observed=True,
)["risk_score"].transform("mean")

df_health[["bmi_category", "risk_score", "avg_risk_score_in_bmi_category"]].head()

,bmi_category,risk_score,avg_risk_score_in_bmi_category
0,obese,3.0,1.672411
1,overweight,1.0,1.433765
2,overweight,2.0,1.433765
3,overweight,1.0,1.433765
4,normal,2.0,1.127319


## 6.2 Create `risk_above_bmi_category_average`.

Compare each respondent's risk score with the average risk score in their BMI category.


In [33]:
df_health["risk_above_bmi_category_average"] = (
    df_health["risk_score"] > df_health["avg_risk_score_in_bmi_category"]
)

df_health["risk_above_bmi_category_average"].value_counts()

risk_above_bmi_category_average
False    142899
True     110781
Name: count, dtype: int64

## 6.3 Add the median BMI within each sex group to every row.

Use `groupby().transform()` to create `median_bmi_by_sex` and then create `bmi_above_sex_median`.


In [34]:
df_health["median_bmi_by_sex"] = df_health.groupby("sex", observed=True)[
    "bmi"
].transform("median")
df_health["bmi_above_sex_median"] = df_health["bmi"] > df_health["median_bmi_by_sex"]

df_health[["sex", "bmi", "median_bmi_by_sex", "bmi_above_sex_median"]].head()

,sex,bmi,median_bmi_by_sex,bmi_above_sex_median
0,female,40.0,27.0,True
1,female,25.0,27.0,False
2,female,28.0,27.0,True
3,female,27.0,27.0,False
4,female,24.0,27.0,False


## 6.4 Compare `agg()` and `transform()`.

Explain the difference in one or two sentences.


`agg()` reduces each group to one output row, so it is useful for summary tables. `transform()` returns one value for each original row, so it is useful when group-level information should be added back to the original DataFrame.


## Huddle after Part 6

> Final checkpoint: The optional challenge should leave `df_health` with row-level columns that compare each respondent to a group benchmark.
